In [47]:
import copy
import json
import math
import os
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler


In [48]:
REQUIRED_CELL_IDS = ['36.50_-119.00', '37.25_-119.00', '38.50_-122.25', '39.00_-121.25', '39.75_-121.00', '40.75_-122.25', '41.75_-123.00', '33.50_-116.25', '37.25_-120.50', '38.75_-123.00', '36.00_-118.25', '39.50_-123.00', '40.75_-123.00', '33.25_-115.50', '34.00_-117.00', '37.75_-120.00', '40.50_-121.25', '41.50_-121.75', '36.25_-120.25', '39.50_-121.25', '40.25_-121.50', '41.00_-122.75', '41.50_-122.00', '39.00_-120.75', '34.25_-118.00', '36.00_-119.00', '36.75_-119.75', '37.00_-120.50', '32.75_-115.50', '34.25_-117.75', '34.25_-119.00', '36.00_-121.00', '36.50_-118.50', '40.75_-123.25', '35.75_-120.75', '36.75_-120.50', '38.75_-121.50', '39.00_-120.50', '41.25_-122.75', '41.75_-123.50', '36.50_-120.00', '38.75_-120.25', '39.75_-121.25', '40.50_-121.50', '40.00_-121.25', '40.25_-120.75', '32.75_-115.75', '38.25_-121.00', '36.25_-119.00', '37.50_-119.25', '37.50_-119.75', '37.75_-119.00', '36.00_-119.25', '38.25_-120.00', '37.25_-119.50', '40.25_-123.25', '41.25_-123.00', '36.00_-121.25', '38.75_-122.50', '34.00_-117.25', '36.75_-119.50', '40.00_-123.00', '37.25_-120.25', '40.50_-123.25', '35.50_-119.25', '36.25_-118.25', '40.25_-122.75', '36.25_-121.25', '36.75_-119.00', '37.50_-119.50', '38.25_-122.00', '37.75_-119.75', '38.00_-119.75', '38.50_-120.50', '38.75_-122.75', '40.00_-120.75', '41.00_-123.00', '33.50_-116.00', '37.50_-121.00', '40.50_-123.75', '36.25_-119.25', '37.25_-120.75', '41.75_-123.25', '36.50_-119.25', '37.25_-119.25', '39.50_-121.50', '40.00_-121.00', '36.75_-118.75', '36.75_-120.25', '40.25_-122.25', '40.75_-123.50', '36.00_-118.50', '37.75_-120.75', '38.00_-121.00', '37.50_-120.50', '41.50_-123.25', '36.25_-118.75', '37.00_-120.00', '37.75_-121.00', '36.75_-120.00', '37.75_-121.50', '40.00_-123.25', '36.50_-118.75', '38.00_-120.00', '40.25_-123.00', '41.00_-123.50', '41.50_-123.50', '36.50_-119.75', '34.00_-117.50', '37.75_-119.50', '38.50_-121.75', '40.00_-122.75', '36.25_-118.50', '41.00_-123.25', '35.75_-119.25', '37.00_-120.25', '40.00_-122.00', '38.50_-122.50', '37.75_-121.25', '38.75_-120.50', '41.25_-123.50', '38.25_-121.50', '38.00_-121.25', '38.25_-121.25', '38.75_-121.75', '37.50_-120.75', '41.25_-123.25', '36.50_-119.50', '39.00_-122.25', '37.00_-119.25', '38.75_-122.00', '39.50_-122.25', '39.50_-121.75', '40.00_-122.25', '39.75_-121.75', '38.50_-122.00', '39.25_-122.00', '39.25_-122.25', '39.00_-121.75', '39.50_-122.00', '39.25_-121.50', '39.00_-121.50', '39.75_-122.25', '39.25_-121.75', '39.00_-122.00', '39.75_-122.00']

In [49]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Dataset / model definitions (unchanged)

In [50]:
class WildfireSequenceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, seq_len, era5_cols, s2_cols, sp5_cols, dem_cols, fire_cols, target_col="y_fire"):
        self.seq_len = seq_len

        df = df.sort_values(["cell_id", "date"], kind="mergesort").reset_index(drop=True)

        self.era5 = df[era5_cols].to_numpy(dtype=np.float32)
        self.s2 = df[s2_cols].to_numpy(dtype=np.float32)
        self.sp5 = df[sp5_cols].to_numpy(dtype=np.float32)
        self.dem = df[dem_cols].to_numpy(dtype=np.float32)
        self.fire = df[fire_cols].to_numpy(dtype=np.float32)
        self.labels = df[target_col].to_numpy(dtype=np.float32)
        self.cell_ids = df["cell_id"].to_numpy()
        self.dates = df["date"].to_numpy()

        cell_arr = self.cell_ids
        n = len(df)
        valid_mask = np.zeros(n, dtype=bool)

        for i in range(seq_len - 1, n):
            if cell_arr[i] == cell_arr[i - (seq_len - 1)]:
                valid_mask[i] = True

        self.target_indices = np.flatnonzero(valid_mask)

    def __len__(self):
        return len(self.target_indices)

    def __getitem__(self, idx):
        t_idx = self.target_indices[idx]
        start = t_idx - self.seq_len + 1
        end = t_idx + 1

        era5_seq = torch.from_numpy(self.era5[start:end])
        s2_seq = torch.from_numpy(self.s2[start:end])
        sp5_seq = torch.from_numpy(self.sp5[start:end])
        dem_val = torch.from_numpy(self.dem[t_idx])
        fire_seq = torch.from_numpy(self.fire[start:end])
        label = torch.tensor(self.labels[t_idx], dtype=torch.float32)

        return (era5_seq, s2_seq, sp5_seq, dem_val, fire_seq), label


In [51]:
class WildfirePredictionEncoderModule(nn.Module):
    def __init__(self, in_dim, out_dim, hidden_mult=2, dropout=0.1):
        super().__init__()
        hidden = out_dim * hidden_mult
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class WildfirePredictionFusionEncoder(nn.Module):
    def __init__(self, era5_emb, s2_emb, sp5_emb, dem_emb, daily_emb):
        super().__init__()
        concat_dim = era5_emb + s2_emb + sp5_emb + dem_emb
        self.fusion_mlp = nn.Sequential(
            nn.Linear(concat_dim, daily_emb),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(daily_emb, daily_emb),
        )

    def forward(self, era5_e, s2_e, sp5_e, dem_e):
        fused = torch.cat([era5_e, s2_e, sp5_e, dem_e], dim=-1)
        return self.fusion_mlp(fused)


class SharedDailyEncoder(nn.Module):
    def __init__(self, era5_dim, era5_emb, s2_dim, s2_emb, sp5_dim, sp5_emb, dem_dim, dem_emb, daily_emb):
        super().__init__()
        self.era5_enc = WildfirePredictionEncoderModule(era5_dim, era5_emb)
        self.s2_enc = WildfirePredictionEncoderModule(s2_dim, s2_emb)
        self.sp5_enc = WildfirePredictionEncoderModule(sp5_dim, sp5_emb)
        self.dem_enc = WildfirePredictionEncoderModule(dem_dim, dem_emb)
        self.fusion = WildfirePredictionFusionEncoder(era5_emb, s2_emb, sp5_emb, dem_emb, daily_emb)

    def forward(self, era5, s2, sp5, dem):
        era5_e = self.era5_enc(era5)
        s2_e = self.s2_enc(s2)
        sp5_e = self.sp5_enc(sp5)
        dem_e = self.dem_enc(dem)
        daily_embedding = self.fusion(era5_e, s2_e, sp5_e, dem_e)
        return daily_embedding

In [52]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


In [53]:
class SharedDailyEncoder(nn.Module):
    def __init__(self, era5_dim, era5_emb, s2_dim, s2_emb, sp5_dim, sp5_emb, dem_dim, dem_emb, fire_dim, fire_emb, out_dim, dropout=0.1):
        super().__init__()
        self.era5_enc = WildfirePredictionEncoderModule(era5_dim, era5_emb, hidden_mult=2, dropout=dropout)
        self.s2_enc = WildfirePredictionEncoderModule(s2_dim, s2_emb, hidden_mult=2, dropout=dropout)
        self.sp5_enc = WildfirePredictionEncoderModule(sp5_dim, sp5_emb, hidden_mult=2, dropout=dropout)
        self.dem_enc = WildfirePredictionEncoderModule(dem_dim, dem_emb, hidden_mult=2, dropout=dropout)
        self.fire_enc = WildfirePredictionEncoderModule(fire_dim, fire_emb, hidden_mult=2, dropout=dropout)

        concat_dim = era5_emb + s2_emb + sp5_emb + dem_emb + fire_emb
        self.fusion = nn.Sequential(
            nn.Linear(concat_dim, out_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, era5_seq, s2_seq, sp5_seq, dem_static, fire_seq):
        B, H, _ = era5_seq.shape
        e_era5 = self.era5_enc(era5_seq.view(B * H, -1)).view(B, H, -1)
        e_s2 = self.s2_enc(s2_seq.view(B * H, -1)).view(B, H, -1)
        e_sp5 = self.sp5_enc(sp5_seq.view(B * H, -1)).view(B, H, -1)
        e_fire = self.fire_enc(fire_seq.view(B * H, -1)).view(B, H, -1)

        e_dem_static = self.dem_enc(dem_static)
        e_dem = e_dem_static.unsqueeze(1).expand(B, H, -1)

        concat = torch.cat([e_era5, e_s2, e_sp5, e_dem, e_fire], dim=-1)
        daily_emb = self.fusion(concat)
        return daily_emb

class WildfirePredictionTransformer(nn.Module):
    def __init__(self, seq_len, n_heads, n_layers, era5_dim, era5_emb, s2_dim, s2_emb, sp5_dim, sp5_emb, dem_dim, dem_emb, fire_dim, fire_emb, daily_emb, ffn_hidden, dropout=0.2):
        super().__init__()
        self.daily_encoder = SharedDailyEncoder(era5_dim, era5_emb, s2_dim, s2_emb, sp5_dim, sp5_emb, dem_dim, dem_emb, fire_dim, fire_emb, daily_emb, dropout=dropout)
        self.pos_encoder = PositionalEncoding(daily_emb, max_len=seq_len, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=daily_emb,
            nhead=n_heads,
            dim_feedforward=ffn_hidden,
            dropout=dropout,
            activation="relu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        self.head = nn.Sequential(
            nn.Linear(daily_emb, daily_emb // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(daily_emb // 2, 1),
        )

    def forward(self, inputs):
        era5_seq, s2_seq, sp5_seq, dem_static, fire_seq = inputs
        daily_emb = self.daily_encoder(era5_seq, s2_seq, sp5_seq, dem_static, fire_seq)
        daily_emb = self.pos_encoder(daily_emb)
        feat = self.transformer(daily_emb)
        last_day = feat[:, -1, :]
        logits = self.head(last_day).squeeze(-1)
        return logits


In [54]:
class HybridFocalRankingLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0, ranking_weight=0.2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ranking_weight = ranking_weight

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        focal_loss = (alpha_t * (1 - p_t) ** self.gamma * bce).mean()

        pos_mask = targets.eq(1)
        neg_mask = targets.eq(0)
        if pos_mask.any() and neg_mask.any():
            pos_logits = logits[pos_mask]
            neg_logits = logits[neg_mask]
            n_pairs = min(len(pos_logits), len(neg_logits), 500)
            pos_sample = pos_logits[:n_pairs]
            neg_sample = neg_logits[:n_pairs]
            target = torch.ones_like(pos_sample)
            margin_loss = F.margin_ranking_loss(pos_sample, neg_sample, target, margin=0.1)
        else:
            margin_loss = 0.0

        return focal_loss + self.ranking_weight * margin_loss


In [55]:
DEFAULT_EVAL_THRESHOLDS = np.round(np.arange(0.10, 0.951, 0.05), 2)


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
    device="cpu",
    thresholds=None,
):
    if thresholds is None:
        thresholds = DEFAULT_EVAL_THRESHOLDS

    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    all_probs, all_labels = [], []
    total_loss = 0.0

    with torch.set_grad_enabled(is_train):
        for era5, s2, sp5, dem, y in loader:
            era5, s2, sp5, dem, y = (
                t.to(device) for t in (era5, s2, sp5, dem, y)
            )

            logits = model(era5, s2, sp5, dem)
            loss = criterion(logits, y)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * y.size(0)

            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.append(y.detach().cpu().numpy())

    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)

    metrics = {
        "loss": total_loss / len(loader.dataset),
        "pr_auc": average_precision_score(labels, probs) if labels.sum() > 0 else float("nan"),
        "roc_auc": roc_auc_score(labels, probs) if labels.sum() > 0 else float("nan"),
        "threshold_metrics": {},
    }

    for t in thresholds:
        preds = (probs >= t).astype(int)

        metrics["threshold_metrics"][round(float(t), 2)] = {
            "recall": recall_score(labels, preds, zero_division=0),
            "precision": precision_score(labels, preds, zero_division=0),
            "f1": f1_score(labels, preds, zero_division=0),
        }

    return metrics

In [56]:
def run_epoch(model, loader, criterion, optimizer=None, device="cuda", thresholds=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_logits = []
    all_targets = []

    with torch.set_grad_enabled(is_train):
        for inputs, targets in loader:
            inputs = [x.to(device) for x in inputs]
            targets = targets.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(inputs)
            loss = criterion(logits, targets)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(targets)
            all_logits.append(logits.detach().cpu())
            all_targets.append(targets.cpu())

    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    all_probs = 1.0 / (1.0 + np.exp(-all_logits))

    avg_loss = total_loss / len(loader.dataset)
    pr_auc = float(average_precision_score(all_targets, all_probs))
    roc_auc = float(roc_auc_score(all_targets, all_probs))

    threshold_recalls = {}
    if thresholds is not None:
        for t in thresholds:
            preds = (all_probs >= t).astype(int)
            pos_mask = all_targets == 1
            rec = float(preds[pos_mask].mean()) if pos_mask.any() else 0.0
            threshold_recalls[t] = rec

    return {
        "loss": avg_loss,
        "pr_auc": pr_auc,
        "roc_auc": roc_auc,
        "probs": all_probs,
        "targets": all_targets,
        "recalls": threshold_recalls,
    }

def train_model(
    model,
    train_loader,
    val_loader,
    focal_alpha,
    focal_gamma,
    lr,
    epochs,
    device,
    thresholds=None,
    early_stopping_patience=5,
    lr_patience=2,
    lr_factor=0.5,
    min_lr=1e-6,
    log_thresholds=(0.1, 0.3, 0.5, 0.8),
):
    if thresholds is None:
        thresholds = DEFAULT_EVAL_THRESHOLDS

    criterion = HybridFocalRankingLoss(alpha=focal_alpha, gamma=focal_gamma, ranking_weight=ranking_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=lr_factor, patience=lr_patience, min_lr=min_lr
    )

    best_val_pr_auc = -1.0
    best_weights = None
    no_improve_epochs = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, device, thresholds)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None, device=device, thresholds=thresholds)

        val_pr_auc = val_metrics["pr_auc"]
        val_roc_auc = val_metrics["roc_auc"]
        scheduler.step(val_pr_auc)

        current_lr = optimizer.param_groups[0]["lr"]

        rec_str = ", ".join([
            f"{t}:{val_metrics['recalls'].get(t, 0.0):.2f}"
            for t in log_thresholds if t in val_metrics['recalls']
        ])

        dt = time.time() - t0
        print(
            f"Epoch {epoch:02d} | train_loss {train_metrics['loss']:.4f} | "
            f"val_loss {val_metrics['loss']:.4f} | val_pr_auc {val_pr_auc:.4f} | "
            f"val_roc_auc {val_roc_auc:.4f} | val_recall[{rec_str}] | "
            f"lr {current_lr:.2e} | time {dt:.1f}s"
        )

        if val_pr_auc > best_val_pr_auc:
            best_val_pr_auc = val_pr_auc
            m_to_save = model.module if isinstance(model, nn.DataParallel) else model
            best_weights = copy.deepcopy(m_to_save.state_dict())
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= early_stopping_patience:
                print(
                    f"Early stopping at epoch {epoch:02d}: no val PR-AUC improvement "
                    f"in {early_stopping_patience} epochs (best={best_val_pr_auc:.4f})."
                )
                break

    if best_weights is not None:
        target_m = model.module if isinstance(model, nn.DataParallel) else model
        target_m.load_state_dict(best_weights)
        print(f"Restored best checkpoint (val PR-AUC = {best_val_pr_auc:.4f}).")

    return model, best_val_pr_auc


## Load data

In [57]:
DATA_DIR = "/kaggle/input/datasets/lakshay654/california-wildfire-knn"

with open(os.path.join(DATA_DIR, "feature_columns.json")) as f:
    raw_feature_columns = json.load(f)

with open(os.path.join(DATA_DIR, "dataset_metadata.json")) as f:
    dataset_metadata = json.load(f)

train_df = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
val_df = pd.read_parquet(os.path.join(DATA_DIR, "val.parquet"))
test_df = pd.read_parquet(os.path.join(DATA_DIR, "test.parquet"))

# Single, shared date-parsing pass -- everything downstream (feature
# engineering, sequence construction, inference) reuses this "date" column
# instead of each consumer re-parsing "feature_end_date"/"date" itself.
for _df in (train_df, val_df, test_df):
    _df["date"] = pd.to_datetime(_df["feature_end_date"])

print("train:", train_df.shape, "| val:", val_df.shape, "| test:", test_df.shape)

train: (981792, 78) | val: (491232, 78) | test: (245280, 78)


In [58]:
def rolling_matrix(arr: np.ndarray, window: int, operation: str) -> np.ndarray:
    frame = pd.DataFrame(arr.T)
    roll = frame.rolling(window=window, min_periods=1)
    if operation == "max":
        out = roll.max()
    elif operation == "min":
        out = roll.min()
    elif operation == "sum":
        out = roll.sum()
    elif operation == "mean":
        out = roll.mean()
    else:
        raise ValueError(f"Unsupported rolling operation: {operation!r}")
    return out.T.to_numpy(dtype="float32")

In [59]:
def rolling_matrix(values: np.ndarray, window: int, operation: str) -> np.ndarray:
    rolling = pd.DataFrame(values.T).rolling(window=window, min_periods=1)
    return getattr(rolling, operation)().to_numpy(dtype="float32").T

def simple_neighbors(frame: pd.DataFrame, cells: int, days: int) -> list[np.ndarray]:
    coordinates = frame.iloc[np.arange(cells) * days][["latitude", "longitude"]].to_numpy(dtype="float64")
    result = []
    for latitude, longitude in coordinates:
        delta_lat = np.abs(coordinates[:, 0] - latitude)
        delta_lon = np.abs(coordinates[:, 1] - longitude)
        mask = (
            (delta_lat <= 0.251)
            & (delta_lon <= 0.251)
            & ~((delta_lat < 1e-9) & (delta_lon < 1e-9))
        )
        result.append(np.flatnonzero(mask))
    return result

def sum_neighbors(values: np.ndarray, neighbors: list[np.ndarray]) -> np.ndarray:
    output = np.zeros_like(values, dtype="float32")
    for index, adjacent in enumerate(neighbors):
        if adjacent.size:
            output[index] = values[adjacent].sum(axis=0)
    return output

def directional_geometry(coordinates: np.ndarray):
    result = []
    for latitude, longitude in coordinates:
        delta_lat = latitude - coordinates[:, 0]
        delta_lon = (longitude - coordinates[:, 1]) * np.cos(np.deg2rad(latitude))
        distance = np.sqrt(delta_lat**2 + delta_lon**2)
        adjacent = np.flatnonzero((distance > 1e-9) & (distance <= 0.36))
        result.append((
            adjacent,
            (delta_lon[adjacent] / distance[adjacent]).astype("float32"),
            (delta_lat[adjacent] / distance[adjacent]).astype("float32"),
            (1.0 / (distance[adjacent] + 0.05)).astype("float32"),
        ))
    return result

def directional_counts(fire: np.ndarray, wind_sin: np.ndarray, wind_cos: np.ndarray, geometry):
    cells, days = fire.shape
    upwind = np.zeros((cells, days), dtype="float32")
    downwind = np.zeros((cells, days), dtype="float32")
    crosswind = np.zeros((cells, days), dtype="float32")
    distance_weighted = np.zeros((cells, days), dtype="float32")
    wind_east, wind_north = -wind_sin, -wind_cos
    for index, (adjacent, east, north, inverse_distance) in enumerate(geometry):
        if not adjacent.size:
            continue
        neighbor_fire = fire[adjacent]
        alignment = wind_east[index][None, :] * east[:, None] + wind_north[index][None, :] * north[:, None]
        upwind[index] = (neighbor_fire * np.maximum(alignment, 0) * inverse_distance[:, None]).sum(axis=0)
        downwind[index] = (neighbor_fire * np.maximum(-alignment, 0) * inverse_distance[:, None]).sum(axis=0)
        crosswind[index] = (
            neighbor_fire * np.sqrt(np.maximum(1.0 - alignment**2, 0)) * inverse_distance[:, None]
        ).sum(axis=0)
        distance_weighted[index] = (neighbor_fire * inverse_distance[:, None]).sum(axis=0)
    return upwind, downwind, crosswind, distance_weighted

def build_features(frame: pd.DataFrame, raw_base_features: list[str]):
    cells = int(frame["cell_id"].nunique())
    days = int(frame["label_date"].nunique())

    # Calendar cycles.
    day_of_year = frame["eo_asof_date"].dt.dayofyear.to_numpy(dtype="float32")
    month = frame["eo_asof_date"].dt.month.to_numpy(dtype="float32")
    calendar = pd.DataFrame({
        "day_of_year_sin": np.sin(2 * np.pi * day_of_year / 365.25),
        "day_of_year_cos": np.cos(2 * np.pi * day_of_year / 365.25),
        "month_sin": np.sin(2 * np.pi * month / 12),
        "month_cos": np.cos(2 * np.pi * month / 12),
    }, index=frame.index).astype("float32")
    frame = pd.concat([frame, calendar], axis=1)

    # Weather physics and history ending at D-5.
    temperature_c = frame["t2m_mean"].to_numpy(dtype="float64") - 273.15
    dewpoint_c = frame["d2m_mean"].to_numpy(dtype="float64") - 273.15
    saturation = 0.6108 * np.exp(17.27 * temperature_c / np.maximum(temperature_c + 237.3, 1e-6))
    actual = 0.6108 * np.exp(17.27 * dewpoint_c / np.maximum(dewpoint_c + 237.3, 1e-6))
    vpd = np.maximum(saturation - actual, 0).astype("float32")
    weather = {
        "vpd_kpa": vpd,
        "vpd_wind_interaction": vpd * frame["wind_speed_mean"].to_numpy(dtype="float32"),
        "vpd_soil_deficit_interaction": vpd * (1 - np.clip(frame["soil_moisture_index"], 0, 1)),
        "heat_soil_deficit_interaction": (
            np.maximum(frame["t2m_max"].to_numpy(dtype="float32") - 273.15, 0)
            * (1 - np.clip(frame["swvl1_mean"], 0, 1))
        ),
        "wind_gust_ratio": frame["i10fg_max"].to_numpy(dtype="float32")
        / (frame["wind_speed_mean"].to_numpy(dtype="float32") + 0.1),
    }
    rolling_specs = {
        "t2m_max": ("max",), "rh_mean": ("min",), "tp_sum_mm": ("sum",),
        "wind_speed_mean": ("max",), "i10fg_max": ("max",),
        "swvl1_mean": ("mean",), "vpd_kpa": ("max", "mean"),
    }
    arrays = {
        name: (weather[name] if name in weather else frame[name].to_numpy(dtype="float32")).reshape(cells, days)
        for name in rolling_specs
    }
    for name, operations in rolling_specs.items():
        for window in (14, 30):
            for operation in operations:
                weather[f"{name}_{operation}_{window}d"] = rolling_matrix(
                    arrays[name], window, operation
                ).reshape(-1)
    temperature = frame["t2m_max"].to_numpy(dtype="float32").reshape(cells, days)
    soil = frame["swvl1_mean"].to_numpy(dtype="float32").reshape(cells, days)
    weather["t2m_max_anomaly_30d"] = (temperature - rolling_matrix(temperature, 30, "mean")).reshape(-1)
    weather["swvl1_anomaly_30d"] = (soil - rolling_matrix(soil, 30, "mean")).reshape(-1)
    weather_frame = pd.DataFrame(weather, index=frame.index).astype("float32")
    frame = pd.concat([frame, weather_frame], axis=1)

    # Fire history ending at D-1 (two target rows behind label day D+1).
    target = frame["y_fire"].to_numpy(dtype="float32").reshape(cells, days)
    lag2 = np.zeros_like(target, dtype="float32")
    lag2[:, 2:] = target[:, :-2]
    history7 = rolling_matrix(lag2, 7, "sum")
    history30 = rolling_matrix(lag2, 30, "sum")
    neighbors = simple_neighbors(frame, cells, days)
    neighbor_lag2 = sum_neighbors(lag2, neighbors)
    neighbor_7d = sum_neighbors(history7, neighbors)
    last_positive = np.full(cells, -10_000, dtype="int32")
    days_since = np.full_like(target, 365, dtype="float32")
    for day in range(days):
        positive = lag2[:, day] > 0
        last_positive[positive] = day
        seen = last_positive > -10_000
        days_since[seen, day] = np.minimum(day - last_positive[seen], 365)
    observed = np.maximum(np.arange(days, dtype="float32") - 1, 0)
    expanding_rate = (np.cumsum(lag2, axis=1, dtype="float32") + 1.0) / (observed[None, :] + 100.0)
    fire = {
        "fire_cell_lag2": lag2.reshape(-1),
        "fire_cell_count_7d_lag2": history7.reshape(-1),
        "fire_cell_count_30d_lag2": history30.reshape(-1),
        "fire_cell_any_7d_lag2": (history7 > 0).astype("float32").reshape(-1),
        "fire_cell_days_since_lag2": days_since.reshape(-1),
        "fire_cell_expanding_rate_lag2": expanding_rate.reshape(-1),
        "fire_neighbor_count_lag2": neighbor_lag2.reshape(-1),
        "fire_neighbor_count_7d_lag2": neighbor_7d.reshape(-1),
        "fire_neighbor_any_7d_lag2": (neighbor_7d > 0).astype("float32").reshape(-1),
        "fire_statewide_cells_7d_lag2": np.tile(history7.sum(axis=0), cells).astype("float32"),
    }
    fire_frame = pd.DataFrame(fire, index=frame.index).astype("float32")
    frame = pd.concat([frame, fire_frame], axis=1)

    # Wind-aware neighboring fire context.
    coordinates = frame.iloc[np.arange(cells) * days][["latitude", "longitude"]].to_numpy(dtype="float64")
    geometry = directional_geometry(coordinates)
    wind_sin = frame["wind_dir_sin"].to_numpy(dtype="float32").reshape(cells, days)
    wind_cos = frame["wind_dir_cos"].to_numpy(dtype="float32").reshape(cells, days)
    upwind_lag2, _, _, distance_lag2 = directional_counts(lag2, wind_sin, wind_cos, geometry)
    upwind_7d, downwind_7d, crosswind_7d, distance_7d = directional_counts(
        history7, wind_sin, wind_cos, geometry
    )
    wind = frame["wind_speed_mean"].to_numpy(dtype="float32")
    vpd = frame["vpd_kpa"].to_numpy(dtype="float32")
    soil_deficit = 1 - np.clip(frame["soil_moisture_index"].to_numpy(dtype="float32"), 0, 1)
    vegetation = np.clip(
        frame["cvh_mean"].to_numpy(dtype="float32") + frame["cvl_mean"].to_numpy(dtype="float32"), 0, 1
    )
    recent_context = np.maximum(
        frame["fire_cell_any_7d_lag2"].to_numpy(dtype="float32"),
        frame["fire_neighbor_any_7d_lag2"].to_numpy(dtype="float32"),
    )
    directional = {
        "fire_upwind_count_lag2": upwind_lag2.reshape(-1),
        "fire_upwind_count_7d_lag2": upwind_7d.reshape(-1),
        "fire_downwind_count_7d_lag2": downwind_7d.reshape(-1),
        "fire_crosswind_count_7d_lag2": crosswind_7d.reshape(-1),
        "fire_distance_weighted_count_lag2": distance_lag2.reshape(-1),
        "fire_distance_weighted_count_7d_lag2": distance_7d.reshape(-1),
        "fire_wind_spread_potential_lag2": upwind_lag2.reshape(-1) * wind,
        "fire_wind_spread_potential_7d_lag2": upwind_7d.reshape(-1) * wind,
        "fire_context_vpd_interaction": frame["fire_neighbor_count_7d_lag2"].to_numpy(dtype="float32") * vpd,
        "fire_context_dry_windy_interaction": recent_context * vpd * wind * soil_deficit,
        "ignition_dry_windy_index": vpd * wind * soil_deficit,
        "fuel_dryness_index": vpd * soil_deficit * vegetation,
        "vpd_short_long_trend": frame["vpd_kpa_mean_14d"] - frame["vpd_kpa_mean_30d"],
        "recent_fire_context": recent_context,
    }
    directional_frame = pd.DataFrame(directional, index=frame.index).astype("float32")
    frame = pd.concat([frame, directional_frame], axis=1)

    # The locked contract excludes a redundant source-availability flag.
    selected_base = [name for name in raw_base_features if name != "s5n_available"]
    feature_columns = list(dict.fromkeys([
        *selected_base,
        "latitude", "longitude",
        *calendar.columns,
        *weather_frame.columns,
        *fire_frame.columns,
        *directional_frame.columns,
    ]))
    groups = {
        "source_after_constant_removal": len(raw_base_features),
        "source_used_by_model": len(selected_base),
        "geographic": 2,
        "calendar": len(calendar.columns),
        "weather_and_interactions": len(weather_frame.columns),
        "causal_fire_history": len(fire_frame.columns),
        "wind_and_context": len(directional_frame.columns),
        "total": len(feature_columns),
    }
    return frame, feature_columns, groups


In [60]:
def prepare_for_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Sort into the [cell_id, eo_asof_date]-major order that build_features()'s
    (cells, days) reshape assumes, and sanity-check the grid is rectangular
    (every cell has the same number of rows) so that reshape is actually valid.
    """
    df = df.sort_values(["cell_id", "eo_asof_date"], kind="mergesort").reset_index(drop=True)
    counts = df.groupby("cell_id", sort=False).size()
    if counts.nunique() > 1:
        raise ValueError(
            "build_features() requires a rectangular (cell x day) grid, but "
            f"per-cell row counts vary: min={counts.min()}, max={counts.max()}. "
            "Filter/align cells to a common date range before calling build_features()."
        )
    return df


target_col = "y_fire"

train_df = train_df[train_df["cell_id"].isin(REQUIRED_CELL_IDS)]

train_df = prepare_for_feature_engineering(train_df)
val_df = prepare_for_feature_engineering(val_df)
test_df = prepare_for_feature_engineering(test_df)

train_df, feature_columns, train_groups = build_features(train_df, raw_feature_columns)
val_df, val_feature_columns, _ = build_features(val_df, raw_feature_columns)
test_df, test_feature_columns, _ = build_features(test_df, raw_feature_columns)

assert feature_columns == val_feature_columns == test_feature_columns, (
    "Engineered feature columns must match exactly across splits."
)

print("Engineered feature groups (train):", train_groups)
print(f"Positives -> train: {train_df[target_col].sum():.0f}/{len(train_df)} "
      f"| val: {val_df[target_col].sum():.0f}/{len(val_df)} "
      f"| test: {test_df[target_col].sum():.0f}/{len(test_df)}")

Engineered feature groups (train): {'source_after_constant_removal': 63, 'source_used_by_model': 62, 'geographic': 2, 'calendar': 4, 'weather_and_interactions': 23, 'causal_fire_history': 10, 'wind_and_context': 14, 'total': 115}
Positives -> train: 9118/213306 | val: 5536/491232 | test: 2275/245280


In [61]:
era5_cols = [
    "t2m_mean", "t2m_max", "t2m_min", "d2m_mean", "rh_mean", "sp_mean",
    "wind_speed_mean", "wind_dir_sin", "wind_dir_cos", "i10fg_max", "tp_sum_mm",
    "swvl1_mean", "swvl2_mean", "soil_moisture_index", "cvh_mean", "cvl_mean",
    "lai_hv_mean", "lai_lv_mean", "blh_mean", "t2m_max_7d", "tp_sum_7d",
    "wind_speed_max_7d", "rh_min_7d", "swvl1_mean_7d", "i10fg_max_7d",
]

dem_cols = ["elevation", "slope", "aspect", "tri", "tpi", "hillshade", "orographic_index"]

s2_cols = [
    "s2n_B2_mean", "s2n_B3_mean", "s2n_B4_mean", "s2n_B8_mean",
    "s2n_B11_mean", "s2n_B12_mean", "s2n_ndvi", "s2n_ndmi", "s2n_nbr", "s2n_available",
]

sp5_cols = [
    "s5n_s5p_aai_mean", "s5n_s5p_co_mean", "s5n_s5p_co_max", "s5n_available",
]

fire_cols = [
    "fire_cell_lag2", "fire_cell_count_7d_lag2", "fire_cell_count_30d_lag2",
    "fire_cell_any_7d_lag2", "fire_cell_days_since_lag2", "fire_cell_expanding_rate_lag2",
    "fire_neighbor_count_lag2", "fire_neighbor_count_7d_lag2", "fire_neighbor_any_7d_lag2",
    "fire_statewide_cells_7d_lag2", "fire_upwind_count_lag2", "fire_upwind_count_7d_lag2",
    "fire_downwind_count_7d_lag2", "fire_crosswind_count_7d_lag2", "fire_distance_weighted_count_lag2",
    "fire_distance_weighted_count_7d_lag2", "fire_wind_spread_potential_lag2",
    "fire_wind_spread_potential_7d_lag2", "fire_context_vpd_interaction", "fire_context_dry_windy_interaction"
]

all_feature_cols = [c for c in feature_columns if c in train_df.columns]

# Ensure every column is accounted for in one group
era5_cols = [c for c in era5_cols if c in all_feature_cols]
dem_cols = [c for c in dem_cols if c in all_feature_cols]
s2_cols = [c for c in s2_cols if c in all_feature_cols]
sp5_cols = [c for c in sp5_cols if c in all_feature_cols]
fire_cols = [c for c in fire_cols if c in all_feature_cols]

# Catch any remaining features into era5_cols
assigned = set(era5_cols + dem_cols + s2_cols + sp5_cols + fire_cols)
unassigned = [c for c in all_feature_cols if c not in assigned]
era5_cols.extend(unassigned)

print(f"era5_cols: {len(era5_cols)} | dem_cols: {len(dem_cols)} | s2_cols: {len(s2_cols)} | sp5_cols: {len(sp5_cols)} | fire_cols: {len(fire_cols)} | total: {len(all_feature_cols)}")


era5_cols: 79 | dem_cols: 6 | s2_cols: 7 | sp5_cols: 3 | fire_cols: 20 | total: 115


In [62]:
scaler = StandardScaler()
train_df[all_feature_cols] = scaler.fit_transform(train_df[all_feature_cols])
val_df[all_feature_cols] = scaler.transform(val_df[all_feature_cols])
test_df[all_feature_cols] = scaler.transform(test_df[all_feature_cols])

train_df[all_feature_cols] = train_df[all_feature_cols].fillna(0.0)
val_df[all_feature_cols] = val_df[all_feature_cols].fillna(0.0)
test_df[all_feature_cols] = test_df[all_feature_cols].fillna(0.0)

In [63]:
seq_len = 7
n_heads = 4
n_layers = 4

era5_dim = len(era5_cols)
era5_emb = 32
s2_dim = len(s2_cols)
s2_emb = 64
sp5_dim = len(sp5_cols)
sp5_emb = 64
dem_dim = len(dem_cols)
dem_emb = 16
fire_dim = len(fire_cols)
fire_emb = 32

daily_emb = 128
ffn_hidden = 512
dropout = 0.2

lr = 1e-4
epochs = 30
batch_size = 64

focal_gamma = 2.0
focal_alpha = 0.5
ranking_weight = 0.2

EARLY_STOPPING_PATIENCE = 5
LR_PATIENCE = 2
LR_FACTOR = 0.5
MIN_LR = 1e-6

EVAL_THRESHOLDS = np.round(np.arange(0.10, 0.951, 0.05), 2)
USE_WEIGHTED_SAMPLER = True
print(f"focal_alpha={focal_alpha:.4f} | focal_gamma={focal_gamma} | ranking_weight={ranking_weight} | sampler={USE_WEIGHTED_SAMPLER}")


focal_alpha=0.5000 | focal_gamma=2.0 | ranking_weight=0.2 | sampler=True


In [64]:
train_ds = WildfireSequenceDataset(train_df, seq_len, era5_cols, s2_cols, sp5_cols, dem_cols, fire_cols)
val_ds = WildfireSequenceDataset(val_df, seq_len, era5_cols, s2_cols, sp5_cols, dem_cols, fire_cols)
test_ds = WildfireSequenceDataset(test_df, seq_len, era5_cols, s2_cols, sp5_cols, dem_cols, fire_cols)
print(f"Sequences -> train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")


Sequences -> train: 212430, val: 487200, test: 241248


In [65]:
def build_train_loader(dataset, batch_size, use_sampler, num_workers=4):
    if use_sampler:
        sample_labels = dataset.labels[dataset.target_indices].astype(int)
        class_counts = np.bincount(sample_labels)
        class_weights = 1.0 / np.maximum(class_counts, 1)
        sample_weights = class_weights[sample_labels]
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
        )
        return DataLoader(
            dataset, batch_size=batch_size, sampler=sampler,
            num_workers=num_workers, pin_memory=True, persistent_workers=True,
        )
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, persistent_workers=True,
    )


train_loader = build_train_loader(train_ds, batch_size, USE_WEIGHTED_SAMPLER)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                         num_workers=4, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

## Model

In [66]:
model = WildfirePredictionTransformer(
    seq_len, n_heads, n_layers,
    era5_dim, era5_emb, s2_dim, s2_emb, sp5_dim, sp5_emb, dem_dim, dem_emb, fire_dim, fire_emb,
    daily_emb, ffn_hidden, dropout=dropout,
).to(device)


In [67]:
n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    print(f"Using {n_gpus} GPUs via nn.DataParallel")
    model = nn.DataParallel(model)
    train_loader = build_train_loader(train_ds, batch_size * n_gpus, USE_WEIGHTED_SAMPLER)

Using 2 GPUs via nn.DataParallel


In [68]:
model, best_val_pr_auc = train_model(
    model,
    train_loader,
    val_loader,
    focal_alpha,
    focal_gamma,
    lr,
    epochs,
    device,
    thresholds=EVAL_THRESHOLDS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    min_lr=MIN_LR,
)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py:134: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return F.linear(input, self.weight, self.bias)


Epoch 01 | train_loss 0.0767 | val_loss 0.0506 | val_pr_auc 0.1922 | val_roc_auc 0.7986 | val_recall[0.1:1.00, 0.3:0.99, 0.5:0.39, 0.8:0.02] | lr 1.00e-04 | time 185.8s
Epoch 02 | train_loss 0.0725 | val_loss 0.0547 | val_pr_auc 0.1971 | val_roc_auc 0.7955 | val_recall[0.1:1.00, 0.3:0.98, 0.5:0.49, 0.8:0.02] | lr 1.00e-04 | time 182.8s
Epoch 03 | train_loss 0.0712 | val_loss 0.0474 | val_pr_auc 0.1931 | val_roc_auc 0.7681 | val_recall[0.1:1.00, 0.3:0.97, 0.5:0.38, 0.8:0.01] | lr 1.00e-04 | time 184.7s
Epoch 04 | train_loss 0.0699 | val_loss 0.0516 | val_pr_auc 0.1899 | val_roc_auc 0.7594 | val_recall[0.1:1.00, 0.3:0.99, 0.5:0.44, 0.8:0.02] | lr 1.00e-04 | time 182.7s
Epoch 05 | train_loss 0.0694 | val_loss 0.0523 | val_pr_auc 0.1799 | val_roc_auc 0.7471 | val_recall[0.1:1.00, 0.3:0.94, 0.5:0.41, 0.8:0.04] | lr 5.00e-05 | time 184.3s
Epoch 06 | train_loss 0.0683 | val_loss 0.0557 | val_pr_auc 0.1846 | val_roc_auc 0.7514 | val_recall[0.1:1.00, 0.3:0.96, 0.5:0.44, 0.8:0.03] | lr 5.00e-05 

In [69]:
@torch.no_grad()
def predict_fire_probability(model, df, cell_id, target_date, seq_len,
                              era5_cols, s2_cols, sp5_cols, dem_cols, device):
    """
    Collects the H days immediately preceding `target_date` for `cell_id`,
    runs the model, and returns P(Fire) on target_date.
    """
    model.eval()
    target_date = pd.to_datetime(target_date)
    # "date" is already parsed to datetime at load time -- no re-conversion needed here.
    cell_df = df[df["cell_id"] == cell_id].sort_values("date")

    window = cell_df[cell_df["date"] < target_date].tail(seq_len)
    if len(window) < seq_len:
        raise ValueError(
            f"Not enough history for cell {cell_id} before {target_date.date()}: "
            f"need {seq_len} days, found {len(window)}."
        )

    def to_tensor(cols):
        arr = window[cols].values.astype(np.float32)
        return torch.from_numpy(arr).unsqueeze(0).to(device)  # (1, T, dim)

    era5 = to_tensor(era5_cols)
    s2 = to_tensor(s2_cols)
    sp5 = to_tensor(sp5_cols)
    dem = to_tensor(dem_cols)

    logit = model(era5, s2, sp5, dem)
    prob = torch.sigmoid(logit).item()
    return prob

In [71]:
# Updated Cell 26 (Test evaluation cell):
test_criterion = HybridFocalRankingLoss(
    alpha=focal_alpha, gamma=focal_gamma, ranking_weight=ranking_weight
)
test_metrics = run_epoch(
    model,
    test_loader,
    test_criterion,
    optimizer=None,
    device=device,
    thresholds=EVAL_THRESHOLDS,
)

print("Test metrics:")
print(f"  loss:    {test_metrics['loss']:.4f}")
print(f"  pr_auc:  {test_metrics['pr_auc']:.4f}")
print(f"  roc_auc: {test_metrics['roc_auc']:.4f}")

recalls = test_metrics["recalls"]
threshold_table = pd.DataFrame(
    [{"threshold": t, "recall": r} for t, r in recalls.items()]
)
print("\nThreshold sweep (Recalls across probability thresholds):")
print(threshold_table.round(4))


Test metrics:
  loss:    0.0556
  pr_auc:  0.1638
  roc_auc: 0.7526

Threshold sweep (Recalls across probability thresholds):
    threshold  recall
0        0.10  1.0000
1        0.15  1.0000
2        0.20  1.0000
3        0.25  0.9956
4        0.30  0.9748
5        0.35  0.9293
6        0.40  0.8272
7        0.45  0.6505
8        0.50  0.3959
9        0.55  0.2589
10       0.60  0.1719
11       0.65  0.1038
12       0.70  0.0857
13       0.75  0.0694
14       0.80  0.0433
15       0.85  0.0057
16       0.90  0.0000
17       0.95  0.0000


## Save model

In [72]:
state_dict = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
torch.save(state_dict, "wildfire_transformer.pt")
print(f"Saved model weights to wildfire_transformer.pt (best val PR-AUC = {best_val_pr_auc:.4f})")

Saved model weights to wildfire_transformer.pt (best val PR-AUC = 0.1971)
